# Install & imports

In [ ]:
!pip install pandas matplotlib numpy -q

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'legend.fontsize': 12,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
})

# Colours — one per run, consistent across all plots
COLOURS = {
    'Baseline':                     '#E07A5F',  # terracotta
    'Best model: Cosine LR + EMA':  '#6B4FBB',  # muted violet
}

def bold_ticks(ax):
    ax.tick_params(axis='both', which='major', labelsize=13, width=1.3)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')

print("Ready.")


# Load CSVs

In [ ]:
DATA_DIR = Path('../data')
if not DATA_DIR.exists():
    DATA_DIR = Path('data')

df_baseline = pd.read_csv(DATA_DIR / 'training_log_cosine_diffusion_scheduler.csv')
df_ema      = pd.read_csv(DATA_DIR / 'training_log_ema.csv')

runs = {
    'Baseline':                    df_baseline,
    'Best model: Cosine LR + EMA': df_ema,
}

for name, df in runs.items():
    print(f"{name}: {len(df)} epochs | "
          f"final loss={df['avg_loss'].iloc[-1]:.5f} | "
          f"FID entries={df['fid'].notna().sum()}")


# Fig 1: Training loss (all runs, full scale)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, df in runs.items():
    ax.plot(df['epoch'], df['avg_loss'],
            color=COLOURS[name], linewidth=1.5, label=name, alpha=0.9)

ax.set_title("Figure 1: Training Loss — All Runs (Full Scale)", fontweight='bold')
ax.set_xlabel("Epoch")
ax.set_ylabel("Average MSE Loss")
ax.legend(prop={'weight': 'bold', 'size': 12})
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
bold_ticks(ax)
plt.tight_layout()
plt.savefig("fig_loss_full.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: fig_loss_full.png")

# Fig 2: Training loss (zoomed in, epoch 50 onwards)

In [ ]:
# Epoch 1 always has a huge spike due to random init — zooming in shows the real trend
fig, ax = plt.subplots(figsize=(10, 5))

for name, df in runs.items():
    df_zoom = df[df['epoch'] >= 50]
    ax.plot(df_zoom['epoch'], df_zoom['avg_loss'],
            color=COLOURS[name], linewidth=1.5, label=name, alpha=0.9)

ax.set_title("Figure 2: Training Loss — Zoomed (Epoch 50 Onwards)", fontweight='bold')
ax.set_xlabel("Epoch")
ax.set_ylabel("Average MSE Loss")
ax.legend(prop={'weight': 'bold', 'size': 12})
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
bold_ticks(ax)
plt.tight_layout()
plt.savefig("fig_loss_zoomed.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: fig_loss_zoomed.png")

# Fig 3: FID over epochs (all runs)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, df in runs.items():
    df_fid = df[df['fid'].notna()]  # Only rows where FID was computed
    df_fid = df_fid[(df_fid['epoch'] >= 100) & (df_fid['epoch'] <= 900)]
    ax.plot(df_fid['epoch'], df_fid['fid'],
            color=COLOURS[name], linewidth=4,
            marker='o', markersize=5, label=name)
    # Annotate best FID
    best_idx = df_fid['fid'].idxmin()
    best_epoch = df_fid.loc[best_idx, 'epoch']
    best_fid   = df_fid.loc[best_idx, 'fid']
    ax.annotate(f"{best_fid:.1f}",
                xy=(best_epoch, best_fid),
                xytext=(0, 12),
                textcoords='offset points',
                color=COLOURS[name], fontsize=13, fontweight='bold',
                ha='center', va='bottom')

ax.set_title("FID Score over Training Epochs", fontweight='bold')
ax.set_xlabel("Epoch")
ax.set_ylabel("FID Score")
ax.legend(prop={'weight': 'bold', 'size': 12})
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
bold_ticks(ax)
plt.tight_layout()
plt.savefig("fig_fid.png", bbox_inches='tight', dpi=250)
plt.show()
print("Saved: fig_fid.png")

# Fig 4: Side-by-side loss + FID (report-ready combined figure)

In [ ]:
# A single combined figure — useful as one appendix figure showing both curves together
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Figure 4: Training Dynamics — Loss and FID", fontweight='bold', fontsize=13)

# Left: loss zoomed
for name, df in runs.items():
    df_zoom = df[df['epoch'] >= 50]
    axes[0].plot(df_zoom['epoch'], df_zoom['avg_loss'],
                 color=COLOURS[name], linewidth=1.5, label=name, alpha=0.9)
axes[0].set_title("Training Loss (Epoch 50+)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Average MSE Loss")
axes[0].legend(prop={'weight': 'bold', 'size': 12})

# Right: FID
for name, df in runs.items():
    df_fid = df[df['fid'].notna()]
    df_fid = df_fid[(df_fid['epoch'] >= 100) & (df_fid['epoch'] <= 900)]
    axes[1].plot(df_fid['epoch'], df_fid['fid'],
                 color=COLOURS[name], linewidth=2,
                 marker='o', markersize=5, label=name)
    best_idx   = df_fid['fid'].idxmin()
    best_epoch = df_fid.loc[best_idx, 'epoch']
    best_fid   = df_fid.loc[best_idx, 'fid']
    axes[1].annotate(f"{best_fid:.1f}",
                     xy=(best_epoch, best_fid),
                     xytext=(0, 10),
                     textcoords='offset points',
                     color=COLOURS[name], fontsize=11, fontweight='bold',
                     ha='center', va='bottom')
axes[1].set_title("FID Score (lower is better)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("FID Score")
axes[1].legend(prop={'weight': 'bold', 'size': 12})

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    bold_ticks(ax)

plt.tight_layout()
plt.savefig("fig_combined.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: fig_combined.png")

# Summary stats table

In [ ]:
rows = []
for name, df in runs.items():
    df_fid = df[df['fid'].notna()]
    best_fid   = df_fid['fid'].min()
    best_epoch = df_fid.loc[df_fid['fid'].idxmin(), 'epoch']
    final_fid  = df_fid['fid'].iloc[-1]
    final_loss = df['avg_loss'].iloc[-1]
    total_epochs = df['epoch'].iloc[-1]
    rows.append({
        'Run':          name,
        'Epochs':       total_epochs,
        'Final Loss':   f"{final_loss:.5f}",
        'Best FID':     f"{best_fid:.2f}",
        'Best FID Epoch': int(best_epoch),
        'Final FID':    f"{final_fid:.2f}",
    })

df_summary = pd.DataFrame(rows).set_index('Run')
print("\n=== Training Summary ===")
print(df_summary.to_string())
df_summary.to_csv("training_summary.csv")
print("\nSaved: training_summary.csv")